In [7]:
import boto3
import json
import uuid
import base64

In [8]:
client = boto3.client("bedrock-agentcore", region_name="us-east-1")

<h2>Basic image agent test</h2>

In [9]:
# --- Point this at your image file ---
IMAGE_PATH = "11.png"  # change to your image path
IMAGE_FORMAT = "png"           # one of: png, jpeg, gif, webp

# Read and base64-encode the image
with open(IMAGE_PATH, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

# Build the payload
image_payload = json.dumps({
    "prompt": "Describe what you see in this image in detail.",
    "image_base64": image_b64,
    "image_format": IMAGE_FORMAT,
})

# Generate a fresh session for the image request
image_session_id = str(uuid.uuid4())

response = client.invoke_agent_runtime(
    agentRuntimeArn='arn:aws:bedrock-agentcore:us-east-1:014498646416:runtime/tom_hackathon-ChO02O61W2',
    runtimeSessionId=image_session_id,
    payload=image_payload,
)

response_body = response['response'].read()
response_data = json.loads(response_body)
print("Agent Response:", response_data)

Agent Response: {'result': {'role': 'assistant', 'content': [{'text': '## Image Description\n\nThis is a **retail advertisement** for men\'s outerwear, likely from a department store.\n\n### The Model:\n- A **young man** with short dark hair, smiling\n- He is **sitting/perching** on a stool\n- He is wearing:\n  - A **dark gray utility/field jacket** with multiple pockets and snap buttons\n  - A **burgundy/dark red shirt** underneath\n  - **Bright orange/rust-colored chinos/pants**\n  - A **brown leather messenger bag** worn across his body\n  - A **brown leather belt**\n\n### The Text:\n- **"STAY WARM IN STYLE"** (large white text)\n- *"shop this season\'s outerwear for men"*\n- **"UP TO 40% OFF"** (in orange/highlighted text) men\'s outerwear\n- Brands mentioned: **Levi\'s®, Tommy Hilfiger®, Guess, Nautica®, Calvin Klein and more!**\n- A **black button** reading **"SHOP ALL OUTERWEAR ▶"**\n\n### Background:\n- **Gray/charcoal textured backdrop**, giving a studio photography feel\n\nTh

<h2>Ranking agent tests</h2>

In [10]:
# Replace with your ranking agent's ARN after deploying to AgentCore
RANKING_AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:014498646416:runtime/ranking_agent-0XoBDAHnnV"

# Example user IAB preference profile (from Stage 1 LR or ground truth)
user_profile = {
    "Automotive": 4.2,
    "Sports": 3.8,
    "Food & Drink": 3.1,
    "Style & Fashion": 2.9,
    "Technology & Computing": 1.1,
    "Health & Fitness": 2.5,
    "Travel": 1.8,
    "Home & Garden": 0.4,
    "Business": 0.2,
    "Education": 0.1,
}

# Example candidate ads with their IAB profiles
candidate_ads = [
    {
        "id": "Cat3_7",
        "iab_profile": {"Sports": 1, "Health & Fitness": 1},
        "description": "Athletic wear ad showing a runner at sunrise",
    },
    {
        "id": "Cat5_2",
        "iab_profile": {"Food & Drink": 1, "Health & Fitness": 1},
        "description": "Organic smoothie brand with vibrant fruit imagery",
    },
    {
        "id": "Cat1_11",
        "iab_profile": {"Automotive": 1, "Technology & Computing": 1},
        "description": "Electric vehicle ad with sleek dashboard close-up",
    },
    {
        "id": "Cat8_4",
        "iab_profile": {"Style & Fashion": 1, "Sports": 1},
        "description": "Athleisure clothing line with urban backdrop",
    },
    {
        "id": "Cat12_9",
        "iab_profile": {"Home & Garden": 1, "Style & Fashion": 1},
        "description": "Modern furniture ad with minimalist living room",
    },
]

ranking_payload = json.dumps({
    "user_profile": user_profile,
    "candidate_ads": candidate_ads,
    "top_k": 5,
})

session_id = str(uuid.uuid4())

response = client.invoke_agent_runtime(
    agentRuntimeArn=RANKING_AGENT_ARN,
    runtimeSessionId=session_id,
    payload=ranking_payload,
)

response_body = response["response"].read()
response_data = json.loads(response_body)

# Pretty-print the ranked results
result = response_data.get("result", response_data)
print(json.dumps(result, indent=2))

{
  "ranked_ads": [
    {
      "rank": 1,
      "id": "Cat1_11",
      "score": 5.3,
      "reasoning": "Despite its pre-sorted score, this ad aligns with the user's dominant Automotive affinity (4.2), which is the single strongest category in the profile. A sleek EV dashboard close-up appeals directly to a car enthusiast's appreciation for design and performance, and the Technology & Computing element (1.1) adds a modest secondary boost. The aspirational, high-tech visual language is likely to generate strong engagement from this user above other ads that split between lower-affinity categories."
    },
    {
      "rank": 2,
      "id": "Cat8_4",
      "score": 6.7,
      "reasoning": "This ad combines Sports (3.8) and Style & Fashion (2.9), two of the user's top categories, creating a meaningful cross-category synergy in the athleisure space. The urban backdrop and lifestyle imagery tap into aspirational aesthetics, and the blend of performance and fashion aligns well with the user